<a href="https://colab.research.google.com/github/saleemproven-jpg/AgenticAi/blob/main/RAG_VectorDB_Responses_API_ChromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Sprint 3 – Enterprise Knowledge Assistant**

**Client:** PROITBRIDGE Operations Manager

### **Business Requirement**
> Our AI Assistant should answer questions from our Employee Handbook PDF.

This notebook teaches the complete RAG workflow from scratch **without LangChain**.


## ====================================================================

# **1. Install Required Libraries**

In [1]:
!pip -q install pypdf openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

## **2. Configure OpenAI API Key**


In [2]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client configured successfully.")

Enter your OpenAI API key: ··········
OpenAI client configured successfully.


## **3. Upload the PDF**


In [3]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))
print("Uploaded file:", pdf_path)

Saving PROITBRIDGE_Employee_Handbook_2026_1.pdf to PROITBRIDGE_Employee_Handbook_2026_1.pdf
Uploaded file: PROITBRIDGE_Employee_Handbook_2026_1.pdf


## **4. PDF Reader**


In [4]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

pages = []
for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    pages.append({
        "page": page_number,
        "text": page_text
    })

text = "\n".join(page["text"] for page in pages)

print("Pages:", len(pages))
print("Characters extracted:", len(text))
print("\nPreview:\n")
print(text[:2000])


Pages: 57
Characters extracted: 104734

Preview:

PEOPLE · POLICIES · PRACTICES
Employee
Handbook
Everything you need to know about working at
PROITBRIDGE — how we hire, how we pay, how we grow,
how we protect our people and our clients, and what we
expect from each other, every single day.
DOCUMENT Employee Handbook — All Employees
VERSION Version 6.0  |  Effective 1 April 2026
APPLIES TO All full-time, part-time, contract and intern staff in India
OWNER Human Resources Department, PROITBRIDGE
REVIEW CYCLE Annually, or on any change in applicable law
CLASSIFICATION Internal — Confidential
PROITBRIDGE TECHNOLOGIES PRIVATE LIMITED PIB-HR-HB-2026-V6
CONTENTS
Table of Contents
This handbook is organised into 22 chapters. Use the page numbers below to navigate to the policy
you need.
01 Welcome to PROITBRIDGE
A message from the Founder & CEO
4
02 About the Company
Who we are, vision, mission, values, structure
6
03 How to Use This Handbook
Scope, authority, amendments, disclaimer
9
04 Empl

## **5. Chunking**

For this example, we use a ***simple character-based chunking strategy***. In a production system, chunking can be made more sophisticated using paragraphs, sections, headings, token counts, or semantic boundaries.


In [5]:
chunk_size = 500
overlap = 50

chunks = []
start = 0
chunk_id = 0

while start < len(text):
    end = start + chunk_size
    chunk_text = text[start:end].strip()

    if chunk_text:
        chunks.append({
            "id": f"chunk-{chunk_id}",
            "text": chunk_text
        })
        chunk_id += 1

    start += chunk_size - overlap

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0]["text"])

Total chunks: 233

First chunk:

PEOPLE · POLICIES · PRACTICES
Employee
Handbook
Everything you need to know about working at
PROITBRIDGE — how we hire, how we pay, how we grow,
how we protect our people and our clients, and what we
expect from each other, every single day.
DOCUMENT Employee Handbook — All Employees
VERSION Version 6.0  |  Effective 1 April 2026
APPLIES TO All full-time, part-time, contract and intern staff in India
OWNER Human Resources Department, PROITBRIDGE
REVIEW CYCLE Annually, or on any change in applica


## **6. Generate Embeddings**

Embeddings convert each chunk of text into a numerical vector that represents its semantic meaning.


In [6]:
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embeddings(texts, batch_size=100):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)

        ordered = sorted(response.data, key=lambda item: item.index)
        all_embeddings.extend([item.embedding for item in ordered])

    return all_embeddings

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = get_embeddings(chunk_texts)

print("Number of embeddings:", len(chunk_embeddings))
print("Embedding dimensions:", len(chunk_embeddings[0]))
print("First 10 values:", chunk_embeddings[0][:10])

Number of embeddings: 233
Embedding dimensions: 1536
First 10 values: [0.010345458984375, 0.0159912109375, 0.051513671875, 0.0234832763671875, -0.00785064697265625, -0.0028324127197265625, 0.0303497314453125, 0.0213775634765625, -0.022491455078125, 0.001178741455078125]


## **7. Store in the Vector Database**

In [7]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="employee_handbook")

# Clear the collection so that re-running this notebook does not create duplicates.
existing = collection.get()
if existing["ids"]:
    collection.delete(ids=existing["ids"])

collection.add(ids=[chunk["id"] for chunk in chunks],
               documents=[chunk["text"] for chunk in chunks],
               embeddings=chunk_embeddings,
               metadatas=[{"source": pdf_path} for _ in chunks])

print("Chunks stored in Vector Database:", collection.count())

Chunks stored in Vector Database: 233


## **8. User Query → Query Embedding**

In [8]:
question = "How many casual leaves are allowed?"

question_embedding = get_embeddings([question])[0]

print("Question:", question)
print("Query embedding dimensions:", len(question_embedding))
print("Query embedding Raw View:", question_embedding)

Question: How many casual leaves are allowed?
Query embedding dimensions: 1536
Query embedding Raw View: [0.044921875, 0.042144775390625, 0.07745361328125, -0.012420654296875, 0.0140380859375, -0.00888824462890625, -0.0239105224609375, -0.032073974609375, 0.00853729248046875, 0.038360595703125, 0.03515625, -0.033782958984375, 0.0285186767578125, 0.0264892578125, 0.015838623046875, -0.0270538330078125, 0.042510986328125, 0.0180206298828125, -0.0026149749755859375, 0.0247650146484375, 0.0219268798828125, 0.059844970703125, 0.005374908447265625, -0.0122222900390625, 0.0305328369140625, -0.0087890625, -0.0128326416015625, -0.05364990234375, 0.032867431640625, -0.018798828125, 0.055389404296875, -0.0211181640625, -0.0017709732055664062, -0.01096343994140625, -0.0249176025390625, -0.0245208740234375, 0.0175323486328125, 0.031005859375, 0.029449462890625, 0.047760009765625, 0.0408935546875, 0.01314544677734375, 0.037353515625, -0.004718780517578125, 0.008056640625, 0.022308349609375, 0.031280

## **9. Retriever → Search the Vector Database**

In [11]:
results

{'ids': [['chunk-70', 'chunk-68', 'chunk-84']],
 'embeddings': None,
 'documents': [['year. Anything above 45 days lapses on 31 December unless\nleave was refused in writing for business reasons.\nEncashed at basic salary on separation, for the balance standing to your credit.\n7.3 Casual Leave (CL)\nFor short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.\nMaximum 3 consecutive days at a time. It cannot be combined with EL.\n• \n• \n• \n• \n• \n• \n• \n• \nPROITBRIDGE Employee Handbook\nVersion 6.0 \xa0|\xa0 Effective 1 April 2026 \xa0|\xa0 Internal & Confidential',
   't On qualifying event Not applicable No\nMarriage Leave 5 days, once in\nservice\nOn qualifying event Not applicable No\nCompensatory Off As earned On approved extra\nwork\n90 days validity No\nLeave Without Pay (LWP) Case by case On approval Not applicable No\nLeave year\nThe leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-\nrated

In [10]:
results = collection.query(query_embeddings=[question_embedding],n_results=3)

retrieved_chunks = results["documents"][0]
retrieved_ids = results["ids"][0]
distances = results["distances"][0]
print("Retrieved chunks:\n")

for i, (chunk_id, chunk_text, distance) in enumerate(zip(retrieved_ids, retrieved_chunks, distances), start=1):
    print(f"--- Result {i} | {chunk_id} | distance={distance:.4f} ---")
    print(chunk_text[:1000])
    print()

Retrieved chunks:

--- Result 1 | chunk-70 | distance=0.9863 ---
year. Anything above 45 days lapses on 31 December unless
leave was refused in writing for business reasons.
Encashed at basic salary on separation, for the balance standing to your credit.
7.3 Casual Leave (CL)
For short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.
Maximum 3 consecutive days at a time. It cannot be combined with EL.
• 
• 
• 
• 
• 
• 
• 
• 
PROITBRIDGE Employee Handbook
Version 6.0  |  Effective 1 April 2026  |  Internal & Confidential

--- Result 2 | chunk-68 | distance=1.0745 ---
t On qualifying event Not applicable No
Marriage Leave 5 days, once in
service
On qualifying event Not applicable No
Compensatory Off As earned On approved extra
work
90 days validity No
Leave Without Pay (LWP) Case by case On approval Not applicable No
Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of

## **10. Prompt + Retrieved Context**

Now we combine the user's question with the relevant chunks returned by the retriever.

This retrieved context becomes part of the input sent to the LLM.


In [12]:
retrieved_chunks

['year. Anything above 45 days lapses on 31 December unless\nleave was refused in writing for business reasons.\nEncashed at basic salary on separation, for the balance standing to your credit.\n7.3 Casual Leave (CL)\nFor short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.\nMaximum 3 consecutive days at a time. It cannot be combined with EL.\n• \n• \n• \n• \n• \n• \n• \n• \nPROITBRIDGE Employee Handbook\nVersion 6.0 \xa0|\xa0 Effective 1 April 2026 \xa0|\xa0 Internal & Confidential',
 't On qualifying event Not applicable No\nMarriage Leave 5 days, once in\nservice\nOn qualifying event Not applicable No\nCompensatory Off As earned On approved extra\nwork\n90 days validity No\nLeave Without Pay (LWP) Case by case On approval Not applicable No\nLeave year\nThe leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-\nrated to the number of completed months remaining in the year.\n7.2 Earned Leave (EL / Priv

In [13]:
context = "\n\n--- Retrieved Chunk ---\n\n".join(retrieved_chunks)

prompt = f"""Answer the user's question using ONLY the retrieved context below.
             If the answer is not present in the context, say that the information is not available in the provided document.
             Retrieved Context:{context}
             User Question:{question}""".strip()

print(prompt)

Answer the user's question using ONLY the retrieved context below.
             If the answer is not present in the context, say that the information is not available in the provided document.
             Retrieved Context:year. Anything above 45 days lapses on 31 December unless
leave was refused in writing for business reasons.
Encashed at basic salary on separation, for the balance standing to your credit.
7.3 Casual Leave (CL)
For short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.
Maximum 3 consecutive days at a time. It cannot be combined with EL.
• 
• 
• 
• 
• 
• 
• 
• 
PROITBRIDGE Employee Handbook
Version 6.0  |  Effective 1 April 2026  |  Internal & Confidential

--- Retrieved Chunk ---

t On qualifying event Not applicable No
Marriage Leave 5 days, once in
service
On qualifying event Not applicable No
Compensatory Off As earned On approved extra
work
90 days validity No
Leave Without Pay (LWP) Case by case On approval Not applicable

## **11. Generate the Answer — OpenAI Responses API**


In [14]:
GENERATION_MODEL = "gpt-4.1-mini"

response = client.responses.create(model=GENERATION_MODEL,input=prompt)
answer = response.output_text
print("Answer:\n")
print(answer)

Answer:

The maximum number of casual leaves (CL) allowed at a time is 3 consecutive days.


# **2 Points to Note**

### **1. The role of the Vector Database?**

- The Vector Database does **not generate the answer**.
- Its job is to **store embeddings and retrieve the most relevant chunks** for a user's query.

### **2.The role of the LLM**

- The LLM receives the retrieved context and generates the final answer.

# **Overall Summary**

- Why RAG exists
- Why a PDF must first be converted into text
- Why Chunking is required
- Why Embeddings are required
- Where the Vector Database fits into the RAG architecture
- How the Retriever searches the Vector Database
- Why retrieved chunks become context for the LLM
- How to generate the final answer using the **OpenAI Responses API**

# **Interview Questions?**